<a href="https://colab.research.google.com/github/TrollRider-Kristian/Springboard-AI-Mini-Projects/blob/main/Student_MLE_MiniProject_Recommendation_Engines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Recommendation Engines

Recommendation engines are algorithms designed to provide personalized suggestions or recommendations to users. These systems analyze user behavior, preferences, and interactions with items (products, movies, music, articles, etc.) to predict and offer items that users are likely to be interested in. Recommendation engines play a crucial role in enhancing user experience, driving engagement, and increasing conversion rates in various applications, including e-commerce, entertainment, content platforms, and more.

There are generally two approaches taken in collaborative filtering and content-based recommendation engines:

**1. Collaborative Filtering:**
Collaborative Filtering is a popular approach to building recommendation systems that leverages the collective behavior of users to make personalized recommendations. It is based on the idea that users who have agreed in the past will likely agree in the future. There are two main types of collaborative filtering:

- **User-based Collaborative Filtering:** This method finds users similar to the target user based on their past interactions (e.g., ratings or purchases). It then recommends items that similar users have liked but the target user has not interacted with yet.

- **Item-based Collaborative Filtering:** In this approach, the system identifies similar items based on user interactions. It recommends items that are similar to the ones the target user has already liked or interacted with.

Collaborative filtering does not require any explicit information about items but relies on the similarity between users or items. It is effective in capturing complex patterns and can provide serendipitous recommendations. However, it suffers from the cold-start problem (i.e., difficulty in recommending to new users or items with no interactions) and scalability challenges in large datasets.

**2. Content-Based Recommendation:**
Content-based recommendation is an alternative approach to building recommendation systems that focuses on the attributes or features of items and users. It leverages the characteristics of items to make recommendations. The key steps involved in content-based recommendation are:

- **Feature Extraction:** For each item, relevant features are extracted. For movies, these features could be genre, director, actors, and plot summary.

- **User Profile:** A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.

- **Similarity Calculation:** The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity or Euclidean distance.

- **Recommendation:** Items that are most similar to the user profile are recommended to the user.

Content-based recommendation systems are less affected by the cold-start problem as they can still recommend items based on their features. They are also more interpretable as they rely on item attributes. However, they may miss out on providing serendipitous recommendations and can be limited by the quality of feature extraction and user profiles.

**Choosing Between Collaborative Filtering and Content-Based:**
Both collaborative filtering and content-based approaches have their strengths and weaknesses. The choice between them depends on the specific requirements of the recommendation system, the type of data available, and the user base. Hybrid approaches that combine collaborative filtering and content-based techniques are also common, aiming to leverage the strengths of both methods and mitigate their weaknesses.

In this mini-project, you'll be building both content based and collaborative filtering engines for the [MovieLens 25M dataset](https://grouplens.org/datasets/movielens/25m/). The MovieLens 25M dataset is one of the most widely used and popular datasets for building and evaluating recommendation systems. It is provided by the GroupLens Research project, which collects and studies datasets related to movie ratings and recommendations. The MovieLens 25M dataset contains movie ratings and other related information contributed by users of the MovieLens website.

**Dataset Details:**
- **Size:** The dataset contains approximately 25 million movie ratings.
- **Users:** It includes ratings from over 162,000 users.
- **Movies:** The dataset consists of ratings for more than 62,000 movies.
- **Ratings:** The ratings are provided on a scale of 1 to 5, where 1 is the lowest rating and 5 is the highest.
- **Timestamps:** Each rating is associated with a timestamp, indicating when the rating was given.

**Data Files:**
The dataset is usually split into three CSV files:

1. **movies.csv:** Contains information about movies, including the movie ID, title, genres, and release year.
   - Columns: movieId, title, genres

2. **ratings.csv:** Contains movie ratings provided by users, including the user ID, movie ID, rating, and timestamp.
   - Columns: userId, movieId, rating, timestamp

3. **tags.csv:** Contains user-generated tags for movies, including the user ID, movie ID, tag, and timestamp.
   - Columns: userId, movieId, tag, timestamp

First, import all the libraries you'll need.

In [1]:
import zipfile
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.metrics.pairwise import cosine_similarity

Next, download the relevant components of the MoveLens dataset. Note, these instructions are roughly based on the colab [here](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/recommendation-systems/recommendation-systems.ipynb?utm_source=ss-recommendation-systems&utm_campaign=colab-external&utm_medium=referral&utm_content=recommendation-systems#scrollTo=O3bcgduFo4s6).

In [2]:
print("Downloading movielens data...")

urlretrieve('http://files.grouplens.org/datasets/movielens/ml-100k.zip', 'movielens.zip')
zip_ref = zipfile.ZipFile('movielens.zip', 'r')
zip_ref.extractall()
print("Done. Dataset contains:")
print(zip_ref.read('ml-100k/u.info'))

ratings_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t', names=ratings_cols, encoding='latin-1')

# The movies file contains a binary feature for each genre.
genre_cols = [
    "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
movies_cols = [
    'movie_id', 'title', 'release_date', "video_release_date", "imdb_url"
] + genre_cols
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', names=movies_cols, encoding='latin-1')

Done. Dataset contains:
b'943 users\n1682 items\n100000 ratings\n'


Before doing any kind of machine learning, it's always good to familiarize yourself with the datasets you'lll be working with.

Here are your tasks:

1. Spend some time familiarizing yourself with both the `movies` and `ratings` dataframes. How many unique user ids are present? How many unique movies are there?
2. Create a new dataframe that merges the `movies` and `ratings` tables on 'movie_id'. Only keep the 'user_id', 'title', 'rating' fields in this new dataframe.

In [3]:
# KRISTIAN_NOTE - Start with a preview of the elements in the movies and ratings dataframes
print("MOVIES DATASET -------")
print(movies.head())
print("RATINGS DATASET -------")
print(ratings.head())

MOVIES DATASET -------
   movie_id              title release_date  video_release_date  \
0         1   Toy Story (1995)  01-Jan-1995                 NaN   
1         2   GoldenEye (1995)  01-Jan-1995                 NaN   
2         3  Four Rooms (1995)  01-Jan-1995                 NaN   
3         4  Get Shorty (1995)  01-Jan-1995                 NaN   
4         5     Copycat (1995)  01-Jan-1995                 NaN   

                                            imdb_url  genre_unknown  Action  \
0  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
1  http://us.imdb.com/M/title-exact?GoldenEye%20(...              0       1   
2  http://us.imdb.com/M/title-exact?Four%20Rooms%...              0       0   
3  http://us.imdb.com/M/title-exact?Get%20Shorty%...              0       1   
4  http://us.imdb.com/M/title-exact?Copycat%20(1995)              0       0   

   Adventure  Animation  Children  ...  Fantasy  Film-Noir  Horror  Musical  \
0          0        

In [4]:
# KRISTIAN_NOTE - I noticed some one-hot encoding for the genre of each film.
# I wonder what all the genres are.  The first five columns are metadata around
# movie_id, title, release_date, video_release_date, and imdb_url.
print(movies.columns[5:])
print("There are " + str(len(movies.columns[5:])) + " distinct genres in the movies dataframe.")

Index(['genre_unknown', 'Action', 'Adventure', 'Animation', 'Children',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')
There are 19 distinct genres in the movies dataframe.


Indices 5 through 23 are genre-specific and one-hot encoded for the movies dataframe.  That means a dataframe consisting of genres alone would be 19 columns long.  This will become relevant later in this project.

In [5]:
# Spend some time familiarizing yourself with both the movies and ratings
# dataframes. How many unique user ids are present? How many unique movies
# are there?
print("The movies dataframe is of size: " + str(movies.shape))
# KRISTIAN_NOTE - Fortunately, Pandas 3 came out with a field that
# counts distinct entries along a column.  See the docs for pd.unique here:
# https://pandas.pydata.org/docs/reference/api/pandas.unique.html#pandas.unique
print("The number of unique movies in the movies dataframe is: " + str(len(pd.unique(movies['movie_id']))))
print("The ratings dataframe is of size: " + str(ratings.shape))
print("The number of unique users in the ratings dataframe is: " + str(len(pd.unique(ratings['user_id']))))
print("The number of unique movies in the ratings dataframe is: " + str(len(pd.unique(ratings['movie_id']))))
print("The number of movie_id values present in both movies and ratings is: " +\
      str(movies['movie_id'].isin(ratings['movie_id']).count()))

The movies dataframe is of size: (1682, 24)
The number of unique movies in the movies dataframe is: 1682
The ratings dataframe is of size: (100000, 4)
The number of unique users in the ratings dataframe is: 943
The number of unique movies in the ratings dataframe is: 1682
The number of movie_id values present in both movies and ratings is: 1682


Although there are 100,000 user reviews in the ratings dataframe, the number of unique movies in both dataframes is the same: 1682.  Since movies are identified by movie_id, and they both possess the same 1682 movie_id values, that means they share the exact same movies, and the merge function is good enough to piece together the two dataframes by movie_id.

In [6]:
# Merge movies and ratings
movies_and_ratings = pd.merge(movies, ratings, on = "movie_id")
# KRISTIAN_NOTE - Yes, the same movie is going to repeat a bunch of times, but
# the user_id and rating is different for each row.  Need a different row for
# each user who reviews the movie because we want it all in the same dataframe.
print(movies_and_ratings.head())
print("The size of the merged dataframe is: " + str(movies_and_ratings.shape))

   movie_id             title release_date  video_release_date  \
0         1  Toy Story (1995)  01-Jan-1995                 NaN   
1         1  Toy Story (1995)  01-Jan-1995                 NaN   
2         1  Toy Story (1995)  01-Jan-1995                 NaN   
3         1  Toy Story (1995)  01-Jan-1995                 NaN   
4         1  Toy Story (1995)  01-Jan-1995                 NaN   

                                            imdb_url  genre_unknown  Action  \
0  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
1  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
2  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
3  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   
4  http://us.imdb.com/M/title-exact?Toy%20Story%2...              0       0   

   Adventure  Animation  Children  ...  Musical  Mystery  Romance  Sci-Fi  \
0          0          1         1  ...        0    

The fact that there are an equal number of rows in the merged dataframe as in the ratings dataframe suggests that the merge function works as intended.

In [39]:
# KRISTIAN_NOTE - Who watched the most movies?  Who watched the fewest?
print(str(len(pd.unique(movies_and_ratings['user_id']))) + " users are represented in the merged dataframe.")
print(movies_and_ratings['user_id'].value_counts())
print("The average number of movies watched per user_id is: " + str(movies_and_ratings['user_id'].value_counts().mean()))
print("The median number of movies watched per user_id is: " + str(movies_and_ratings['user_id'].value_counts().median()))

943 users are represented in the merged dataframe.
user_id
405    737
655    685
13     636
450    540
276    518
      ... 
36      20
364     20
685     20
631     20
418     20
Name: count, Length: 943, dtype: int64
The average number of movies watched per user_id is: 106.04453870625663
The median number of movies watched per user_id is: 65.0


User 405 watched 737 movies while user 418 watched only 20.  All 943 unique users are represented in the merged dataframe.  The average number of movies seen by a user is approximately 106.  The median movie watcher has seen 65 movies from this dataset.

In [8]:
# KRISTIAN_NOTE - This helper function will help both content-based filtering AND
# collaborative filtering.
def get_rated_movies_by_user(df, user_id):
  return df.loc[df['user_id'] == user_id, :]

rated_movies_308 = get_rated_movies_by_user(movies_and_ratings, 308)
print(rated_movies_308.loc[:, ['user_id', 'title', 'rating']].head())
print("User 308 has seen and rated: " + str(rated_movies_308.shape[0]) + " movies.")
rated_movies_247 = get_rated_movies_by_user(movies_and_ratings, 247)
print(rated_movies_247.loc[:, ['user_id', 'title', 'rating']].head())
print("User 247 has seen and rated: " + str(rated_movies_247.shape[0]) + " movies.")
rated_movies_148 = get_rated_movies_by_user(movies_and_ratings, 148)
print(rated_movies_148.loc[:, ['user_id', 'title', 'rating']].head())
print("User 148 has seen and rated: " + str(rated_movies_148.shape[0]) + " movies.")

      user_id                  title  rating
0         308       Toy Story (1995)       4
681       308      Get Shorty (1995)       5
889       308         Copycat (1995)       4
1309      308  Twelve Monkeys (1995)       4
1471      308            Babe (1995)       5
User 308 has seen and rated: 397 movies.
      user_id                  title  rating
52        247       Toy Story (1995)       4
1241      247  Twelve Monkeys (1995)       4
4651      247       Apollo 13 (1995)       5
6453      247       Star Wars (1977)       5
8109      247       Quiz Show (1994)       4
User 247 has seen and rated: 26 movies.
      user_id                  title  rating
2         148       Toy Story (1995)       4
1053      148  Twelve Monkeys (1995)       5
1502      148            Babe (1995)       4
6535      148       Star Wars (1977)       5
7719      148    Pulp Fiction (1994)       5
User 148 has seen and rated: 65 movies.


In [9]:
# KRISTIAN_NOTE - Given a user_id, this helper function will return all the movies
# a person has NOT seen.  It will help with both content-based filtering AND
# collaborative filtering.
def filter_out_already_seen_movies(df, super_df, user_id, category):
  if (category == 'title'):
    rated_titles = list(get_rated_movies_by_user(super_df, user_id)['title'])
    return df.loc[~df['title'].isin(rated_titles)]
  else:
    rated_titles = list(get_rated_movies_by_user(super_df, user_id).index)
    return df.loc[~df.index.isin(rated_titles)]

unrated_movies_308 = filter_out_already_seen_movies(movies, movies_and_ratings, 308, 'title')
print("User 308 still has " + str(unrated_movies_308.shape[0]) + " movies left to see.")
unrated_movies_247 = filter_out_already_seen_movies(movies, movies_and_ratings, 247, 'title')
print("User 247 still has " + str(unrated_movies_247.shape[0]) + " movies left to see.")
unrated_movies_148 = filter_out_already_seen_movies(movies, movies_and_ratings, 148, 'title')
print("User 148 still has " + str(unrated_movies_148.shape[0]) + " movies left to see.")

User 308 still has 1284 movies left to see.
User 247 still has 1656 movies left to see.
User 148 still has 1617 movies left to see.


As mentioned in the introduction, content-Based Filtering is a recommendation engine approach that focuses on the attributes or features of items (products, movies, music, articles, etc.) and leverages these features to make personalized recommendations. The underlying idea is to match the characteristics of items with the preferences of users to suggest items that align with their interests. Content-based filtering is particularly useful when explicit user-item interactions (e.g., ratings or purchases) are sparse or unavailable.

**Key Steps in Content-Based Filtering:**

1. **Feature Extraction:**
   - For each item, relevant features are extracted. These features are typically descriptive attributes that can be represented numerically, such as genre, director, actors, author, publication date, and keywords.
   - In the case of text-based items, natural language processing techniques may be used to extract features like TF-IDF (Term Frequency-Inverse Document Frequency) scores.

2. **User Profile Creation:**
   - A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.
   - For example, if a user has watched several action movies, the action genre feature would receive a higher weight in their profile.

3. **Similarity Calculation:**
   - The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity, Euclidean distance, or Pearson correlation.
   - Cosine similarity is commonly used as it measures the cosine of the angle between two vectors, which represents their similarity.

4. **Recommendation:**
   - Items that are most similar to the user profile are recommended to the user. These are items whose features have the highest similarity scores with the user profile.
   - The recommended items are presented as a list sorted by their similarity scores.

**Advantages of Content-Based Filtering:**
1. **No Cold-Start Problem:** Content-based filtering can make recommendations even for new users with no historical interactions because it relies on item features rather than user history.

2. **User Independence:** The recommendations are based solely on the features of items and do not require knowledge of other users' preferences or behavior.

3. **Transparency:** Content-based recommendations are interpretable, as they depend on the features of items, making it easier for users to understand why specific items are recommended.

4. **Serendipity:** Content-based filtering can recommend items with characteristics not seen before by the user, leading to serendipitous discoveries.

5. **Diversity in Recommendations:** The method can offer diverse recommendations since it suggests items with different feature combinations.

**Limitations of Content-Based Filtering:**
1. **Limited Discovery:** Content-based filtering may struggle to recommend items outside the scope of users' historical interactions or interests.

2. **Over-Specialization:** Users may receive recommendations that are too similar to their previous choices, leading to a lack of exposure to new item categories.

3. **Dependency on Feature Quality:** The quality and relevance of item features significantly influence the quality of recommendations.

4. **Limited for Cold Items:** Content-based filtering can struggle to recommend new items with limited feature information.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return content-based recommendations for this user. Here are steps you can take:

  A. Get the user's rated movies

  B. Create a TF-IDF matrix using movie genres. Note, this can be extracted from the `movies` dataframe.

  C. Compute the cosine similarity between movie genres. Use the [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) function.

  D. Get the indices of similar movies to those rated by the user based on cosine similarity. Keep only the top 5.

  E. Remove duplicates and movies already rated by the user.

In [10]:
# STEP_1 - Create a TF-IDF matrix using movie genres.
# The kicker is that since the movie genres are already one-hot encoded,
# we don't need a TF-IDF vectorizer because the movies are already "vectorized".
def get_genres_only_matrix_by_user_id (df, user_id):
  genres_list = list(df.columns[5:23])
  genres_matrix = get_rated_movies_by_user(df, user_id).loc[:, genres_list]
  genres_matrix.index = df.loc[df['user_id'] == user_id, 'title']
  return genres_matrix

genres_only_df = get_genres_only_matrix_by_user_id(movies_and_ratings, 308)
print(genres_only_df.head())
print("And the size of the genres only dataframe is: " + str(genres_only_df.shape))

                       genre_unknown  Action  Adventure  Animation  Children  \
title                                                                          
Toy Story (1995)                   0       0          0          1         1   
Get Shorty (1995)                  0       1          0          0         0   
Copycat (1995)                     0       0          0          0         0   
Twelve Monkeys (1995)              0       0          0          0         0   
Babe (1995)                        0       0          0          0         1   

                       Comedy  Crime  Documentary  Drama  Fantasy  Film-Noir  \
title                                                                          
Toy Story (1995)            1      0            0      0        0          0   
Get Shorty (1995)           1      0            0      1        0          0   
Copycat (1995)              0      1            0      1        0          0   
Twelve Monkeys (1995)       0      0   

In [11]:
# STEP_1.5 - On a similar note, we also need the genres of the movies the user
# has NOT seen yet.  This will be for the purposes of cosine similarity.
def get_unseen_genres_matrix_by_user_id(df, super_df, user_id):
  genres_list = list(df.columns[5:23])
  genres_matrix = filter_out_already_seen_movies(df, super_df, user_id, 'title')
  genres_matrix.index = genres_matrix.loc[:, 'title']
  return genres_matrix.loc[:, genres_list]

unseen_genres_df = get_unseen_genres_matrix_by_user_id(movies, movies_and_ratings, 308)
print(unseen_genres_df.head())
print("And the size of the genres only dataframe is: " + str(unseen_genres_df.shape))

                                                    genre_unknown  Action  \
title                                                                       
GoldenEye (1995)                                                0       1   
Four Rooms (1995)                                               0       0   
Shanghai Triad (Yao a yao yao dao waipo qiao) (...              0       0   
Richard III (1995)                                              0       0   
Mighty Aphrodite (1995)                                         0       0   

                                                    Adventure  Animation  \
title                                                                      
GoldenEye (1995)                                            1          0   
Four Rooms (1995)                                           0          0   
Shanghai Triad (Yao a yao yao dao waipo qiao) (...          0          0   
Richard III (1995)                                          0          0   
Migh

If we take cosine similarity now, we'd know how similar any of the user's already seen movies is to any of the unseen movies, but this does not take rating into account.  We need a user profile, and to compute that, I've decided to take the weighted average of each genre.  The more high-rated movies a user has seen in a particular genre, the higher the score, and that should alter the results of the upcoming cosine similarity matrix more towards movies the user has enjoyed.

In [12]:
# STEP_2 - Multiply each of the user's already seen movies by the rating and
# obtain a weighted average of each genre.
def scale_genres_by_user_ratings(df, user_id):
  ratings_vector = list(get_rated_movies_by_user(df, user_id).loc[:, 'rating'])
  genres_matrix = get_genres_only_matrix_by_user_id(df, user_id)
  # KRISTIAN_NOTE - mul works ONLY because I intentionally leave the movie titles
  # in the same order in both the ratings vector and the genre matrix
  return genres_matrix.mul(ratings_vector, axis=0)

print(get_rated_movies_by_user(movies_and_ratings, 308).loc[:, ['title', 'rating']].head())
print(scale_genres_by_user_ratings(movies_and_ratings, 308).head())

                      title  rating
0          Toy Story (1995)       4
681       Get Shorty (1995)       5
889          Copycat (1995)       4
1309  Twelve Monkeys (1995)       4
1471            Babe (1995)       5
                       genre_unknown  Action  Adventure  Animation  Children  \
title                                                                          
Toy Story (1995)                   0       0          0          4         4   
Get Shorty (1995)                  0       5          0          0         0   
Copycat (1995)                     0       0          0          0         0   
Twelve Monkeys (1995)              0       0          0          0         0   
Babe (1995)                        0       0          0          0         5   

                       Comedy  Crime  Documentary  Drama  Fantasy  Film-Noir  \
title                                                                          
Toy Story (1995)            4      0            0      0       

In [13]:
# STEP_3 - Get the mean genre by user_id
def get_mean_genres_by_user(df, user_id):
  return scale_genres_by_user_ratings(df, user_id).mean()

test_mean_genres_user_308 = get_mean_genres_by_user(movies_and_ratings, 308)
print(test_mean_genres_user_308)

genre_unknown    0.000000
Action           0.690176
Adventure        0.390428
Animation        0.123426
Children         0.234257
Comedy           1.226700
Crime            0.367758
Documentary      0.055416
Drama            1.518892
Fantasy          0.062972
Film-Noir        0.138539
Horror           0.251889
Musical          0.267003
Mystery          0.181360
Romance          0.612091
Sci-Fi           0.385390
Thriller         0.773300
War              0.284635
dtype: float64


Since we already have a vector of 1's and 0's for the genres of each movie title, cosine similarity will work on the movies matrix as-is without the need for a TF-IDF vectorizer.  For example, "Goldeneye" and "Copycat" share 1 genre in common (Thriller) and receive a similarity score of 0.33333.

In [17]:
# STEP_4 - Get the indices of the similar movies based on cosine similarity
# Do this on the scaled average of user-genre preferences we extracted in STEP_3.
def get_cosine_similarity_by_genre(df, super_df, user_id):
  mean_preferences = get_mean_genres_by_user(super_df, user_id)
  unseen_genres_matrix = get_unseen_genres_matrix_by_user_id(df, super_df, user_id)
  similarity_array = cosine_similarity(mean_preferences.values.reshape(1, -1), unseen_genres_matrix)
  return pd.DataFrame(similarity_array.T, index=unseen_genres_matrix.index, columns=["similarity_score"])

print(list(get_cosine_similarity_by_genre(movies, movies_and_ratings, 308).sort_values(by=['similarity_score'], ascending = False).head().index))

['House of Yes, The (1997)', 'Swimming with Sharks (1995)', 'Ma vie en rose (My Life in Pink) (1997)', 'Twisted (1996)', 'Big Bully (1996)']


In [18]:
# Content-Based Filtering using Movie Genres
def content_based_recommendation(user_id, df):
  # Get the user's rated movies
  # Create a TF-IDF matrix using movie genres
  # Compute the cosine similarity between movie genres
  # Get the indices of the similar movies to those rated by the user based on cosine similarity
  # Remove duplicates and movies already rated by the user
  return list(get_cosine_similarity_by_genre(movies, df, user_id).sort_values(by=['similarity_score'], ascending = False).head().index)

The key idea behind collaborative filtering is that users who have agreed in the past will likely agree in the future. Instead of relying on item attributes or user profiles, collaborative filtering identifies patterns of user behavior and item preferences from the interactions present in the data.

**Types of Collaborative Filtering:**
There are two main types of collaborative filtering:

**Collaborative Filtering Process:**
The collaborative filtering process typically involves the following steps:

1. **Data Collection:**
   - Gather data on user-item interactions, such as movie ratings, product purchases, or article clicks.

2. **User-Item Matrix:**
   - Organize the data into a user-item matrix, where rows represent users, columns represent items, and the entries contain the users' interactions (e.g., ratings).

3. **Similarity Calculation:**
   - Calculate the similarity between users or items using similarity metrics such as cosine similarity, Pearson correlation, or Jaccard similarity.
   - For user-based collaborative filtering, user similarities are calculated, and for item-based collaborative filtering, item similarities are calculated.

4. **Neighborhood Selection:**
   - For each user or item, select the most similar users or items as the neighborhood.
   - The size of the neighborhood (the number of similar users or items to consider) is an important parameter to control the system's behavior.

5. **Prediction Generation:**
   - Predict the ratings for items that the target user has not yet interacted with by combining the ratings of neighboring users or items.

6. **Recommendation Generation:**
   - Recommend items with the highest predicted ratings to the target user.

**Advantages of Collaborative Filtering using User-Item Interactions:**
- Collaborative filtering is based solely on user interactions and does not require knowledge of item attributes, making it useful for cases where item data is sparse or unavailable.
- It can provide serendipitous recommendations, suggesting items that users may not have discovered on their own.
- Collaborative filtering can be applied in various domains, including e-commerce, music, movie, and content recommendations.

**Limitations of Collaborative Filtering:**
- The cold-start problem: Collaborative filtering struggles to recommend to new users or items with no or limited interaction history.
- It may suffer from sparsity when data is limited or when users have only interacted with a small subset of items.
- Scalability issues can arise with large datasets and an increasing number of users or items.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return collaborative filtering recommendations for this user based on a user-item interaction matrix. Here are steps you can take:

  A. Create the user-item matrix using Pandas' [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).

  B. Fill missing values with zeros in this matrix.

  C. Calculate user-user similarity matrix using cosine similarity.

  D. Get the array of similarity scores of the target user with all other users from the similarity matrix.

  E. Extract, say the the top 5 most similar users (excluding the target user).

  F. Generate movie recommendations based on the most similar users.

  G. Remove duplicate movies recommendations.

In [19]:
# STEP_1 - The pivot_table function also accepts a fill value as an extra parameter,
# so I can fill all the missing ratings with 0 in the same function call.
def get_users_and_titles_df (df):
  return pd.pivot_table(df, values='rating', index='user_id', columns='title', fill_value=0)

users_and_titles_df = get_users_and_titles_df(movies_and_ratings)
print(users_and_titles_df.head())

title    'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
user_id                                                                   
1                              0.0           0.0                    2.0   
2                              0.0           0.0                    0.0   
3                              0.0           0.0                    0.0   
4                              0.0           0.0                    0.0   
5                              0.0           0.0                    2.0   

title    12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
user_id                                                                 
1                        5.0         0.0                          0.0   
2                        0.0         0.0                          0.0   
3                        0.0         2.0                          0.0   
4                        0.0         0.0                          0.0   
5                        0.0        

In [20]:
# STEP_2 - Get the cosine similarity matrix comparing the movie tastes of two different users.
# It's based on the user-rating table computed above.
def get_user_user_similarity_matrix(df):
  return pd.DataFrame(cosine_similarity(df), index=df.index, columns=df.index)

user_user_similarity_df = get_user_user_similarity_matrix(users_and_titles_df)
print(user_user_similarity_df.head())
print(user_user_similarity_df.shape)

user_id       1         2         3         4         5         6         7    \
user_id                                                                         
1        1.000000  0.168937  0.048388  0.064561  0.379670  0.429682  0.443097   
2        0.168937  1.000000  0.113393  0.179694  0.073623  0.242106  0.108604   
3        0.048388  0.113393  1.000000  0.349781  0.021592  0.074018  0.067423   
4        0.064561  0.179694  0.349781  1.000000  0.031804  0.068431  0.091507   
5        0.379670  0.073623  0.021592  0.031804  1.000000  0.238636  0.374733   

user_id       8         9         10   ...       934       935       936  \
user_id                                ...                                 
1        0.320079  0.078385  0.377733  ...  0.372213  0.119860  0.269860   
2        0.104257  0.162470  0.161273  ...  0.147095  0.310661  0.363328   
3        0.084419  0.062039  0.066217  ...  0.033885  0.043453  0.167140   
4        0.188060  0.101284  0.060859  ...  0.054615

Note that the cosine similarity matrix is 943 x 943.  This is the same number of rows and columns as the number of unique user id's in the original movies and ratings datasets.  For a given user, we need only lookup the row corresponding to their id and we can then sort the cosine similarities of all other users.

In [21]:
# STEP_3 - Given a specific user_id, sort all the other user_id's in descending order and pull out the top 5.
def get_top_five_similar_users(df, user_id):
  # Do a slice of 1: because we want the top 5 user_id's that aren't the same as the original.
  return df.loc[:, user_id].sort_values(ascending = False)[1:].head()

test_similarities_user_308 = get_top_five_similar_users(user_user_similarity_df, 308)
print("The users most similar to user 308 are: " + str(list(test_similarities_user_308.index)))
test_similarities_user_234 = get_top_five_similar_users(user_user_similarity_df, 234)
print("The users most similar to user 234 are: " + str(list(test_similarities_user_234.index)))
test_similarities_user_474 = get_top_five_similar_users(user_user_similarity_df, 474)
print("The users most similar to user 474 are: " + str(list(test_similarities_user_474.index)))

The users most similar to user 308 are: [234, 293, 59, 7, 429]
The users most similar to user 234 are: [474, 308, 537, 716, 7]
The users most similar to user 474 are: [537, 271, 234, 59, 716]


It makes sense that two users are similar to each other, so user 234 appears in the top five similar users to user 308 and vice-versa.  Same goes for user 234 and user 474.  But it's also a good sign that there are enough distinct users to keep the algorithm interesting.

Then, we take the users and titles DataFrame, remove all the movies that the current user in question has already seen, and filter out all the rows except the most similar five computed above.  Take the mean rating of each one to obtain the expected rating of each unseen movie title.  Sort these ratings in descending order, and the top 5 are the ones we will recommend to that specific user.

In [26]:
# STEP_4 - Take the average cosine similarity of the five scores generated from
# the top 5 similar users as a prediction for how much the specific user will
# enjoy each movie title.  Sort and take the top 5.
def recommend_movies_based_on_similar_users(df, users_list):
  return list(df.loc[df.index.isin(users_list), :].mean().sort_values(ascending = False).head().index)

test_seen_movies_user_308 = filter_out_already_seen_movies(users_and_titles_df, movies_and_ratings, 308, 'index')
print(recommend_movies_based_on_similar_users(test_seen_movies_user_308, list(test_similarities_user_308.index)))

['Star Wars (1977)', 'Bridge on the River Kwai, The (1957)', 'Citizen Kane (1941)', 'Casablanca (1942)', 'Maltese Falcon, The (1941)']


In [27]:
# Collaborative Filtering using User-Item Interactions
def collaborative_filtering_recommendation(user_id, df):
  # Create the user-item matrix
  # Fill missing values with 0 (indicating no rating)
  user_item_matrix = get_users_and_titles_df(df)
  # Calculate user-user similarity matrix using cosine similarity
  similarity_matrix = get_user_user_similarity_matrix(user_item_matrix)
  # Get the similarity scores of the target user with all other users
  # Find the top N most similar users (excluding the target user)
  similar_users = get_top_five_similar_users(similarity_matrix, user_id)
  # Generate movie recommendations based on the most similar users
  # Remove duplicates from recommendations
  filtered_movies = filter_out_already_seen_movies(user_item_matrix, df, user_id, 'index')
  return recommend_movies_based_on_similar_users(filtered_movies, list(similar_users.index))

Now, test your recommendations engines! Select a few user ids and generate recommendations using both functions you've written. Are the recommendations similar? Do the recommendations make sense?

In [57]:
# Test the recommendation engines - Content-based Recommendation
# Collaborative-filtering recommendations will be tested in the next code cell.

# Test user 148
print('USER 148 ----------')
content_based_test_user_148 = content_based_recommendation(148, movies_and_ratings)
print(content_based_test_user_148)
already_seen_movies_148 = get_rated_movies_by_user(movies_and_ratings, 148)
print("User 148 has seen: " + str(already_seen_movies_148.loc[already_seen_movies_148['title'].isin(content_based_test_user_148)].shape[0]) + " of these movies.")
print(get_mean_genres_by_user(movies_and_ratings, 148).sort_values(ascending = False).head())
unseen_genre_matrix_user_148 = get_unseen_genres_matrix_by_user_id(movies, movies_and_ratings, 148)
print(unseen_genre_matrix_user_148.loc[unseen_genre_matrix_user_148.index.isin(content_based_test_user_148)])

# Test user 405
print('USER 405 ----------')
content_based_test_user_405 = content_based_recommendation(405, movies_and_ratings)
print(content_based_test_user_405)
already_seen_movies_405 = get_rated_movies_by_user(movies_and_ratings, 405)
print("User 405 has seen: " + str(already_seen_movies_405.loc[already_seen_movies_405['title'].isin(content_based_test_user_405)].shape[0]) + " of these movies.")
print(get_mean_genres_by_user(movies_and_ratings, 405).sort_values(ascending = False).head())
unseen_genre_matrix_user_405 = get_unseen_genres_matrix_by_user_id(movies, movies_and_ratings, 405)
print(unseen_genre_matrix_user_405.loc[unseen_genre_matrix_user_405.index.isin(content_based_test_user_405)])

# Test user 405
print('USER 418 ----------')
content_based_test_user_418 = content_based_recommendation(418, movies_and_ratings)
print(content_based_test_user_418)
already_seen_movies_418 = get_rated_movies_by_user(movies_and_ratings, 418)
print("User 418 has seen: " + str(already_seen_movies_148.loc[already_seen_movies_148['title'].isin(content_based_test_user_148)].shape[0]) + " of these movies.")
print(get_mean_genres_by_user(movies_and_ratings, 418).sort_values(ascending = False).head())
unseen_genre_matrix_user_418 = get_unseen_genres_matrix_by_user_id(movies, movies_and_ratings, 418)
print(unseen_genre_matrix_user_418.loc[unseen_genre_matrix_user_418.index.isin(content_based_test_user_418)])

USER 148 ----------
['Stand by Me (1986)', 'Pollyanna (1960)', 'Faster Pussycat! Kill! Kill! (1965)', 'Get Shorty (1995)', 'Aladdin (1992)']
User 148 has seen: 0 of these movies.
Drama        1.415385
Comedy       1.369231
Animation    0.953846
Adventure    0.907692
Sci-Fi       0.892308
dtype: float64
                                     genre_unknown  Action  Adventure  \
title                                                                   
Get Shorty (1995)                                0       1          0   
Faster Pussycat! Kill! Kill! (1965)              0       1          0   
Aladdin (1992)                                   0       0          0   
Stand by Me (1986)                               0       0          1   
Pollyanna (1960)                                 0       0          0   

                                     Animation  Children  Comedy  Crime  \
title                                                                     
Get Shorty (1995)                 

The tests above run for content-based recommendation.  I tested three user id's: a median movie viewer (user 148 who has seen 65 movies), the user who has seen the most movies (user 405 with a grand total of 737 movies), and one of the users who has seen the least (user 418 who has seen only 20 movies).

User 148 has a diverse spread of genre preferences with drama and comedy reigning supreme but with some animation, adventure, and sci-fi preferences sprinkled in.  The top recommendation, "Stand By Me", is a drama and a comedy and an adventure.  All five recommendations are comedies and all except the fifth recommendation ("Aladdin") are dramas.  Aladdin is an animation, so all these movies have common genres but there is enough diversity to keep user 148 engaged.  Content-based recommendation makes sense for user 148.

User 405's top three genres are calculated to be drama, comedy, and action.  All five movie recommendations are both dramas and comedies.  The third preferred genre of user 405 is "Action", while the top movie recommendation ("Faster Pussycat! Kill! Kill!") is "Action" in addition to being a drama and a comedy.  Although there is a lack of genre diversity in the top five recommendations, all five of them are in line with user 405's movie preferences.  Content-based recommendation makes sense for user 405.

User 418 has a strong preference for thrillers and dramas with a distant third preference for Mystery movies.  Looking at the recommendations, all five movies are both dramas and thrillers, while the top recommendation ("The Client") is a mystery in addition to the other two genres.  Again, despite the lack of genre diversity in the top five recommendations, all five movies are in line with user 418's movie preferences.  Content-based recommendation makes sense for user 418.

In [54]:
# Test the recommendation engines - Collaborative-filtering Recommendation
def display_similar_users_to_user_id(df, user_id):
  user_item_matrix = get_users_and_titles_df(df)
  similarity_matrix = get_user_user_similarity_matrix(user_item_matrix)
  return list(get_top_five_similar_users(similarity_matrix, user_id).index)

def check_ratings_by_similar_users(df, user_id, similar_users, movie_list):
  return df.loc[df['user_id'].isin(similar_users) & df['title'].isin(movie_list), ['title', 'user_id', 'rating']]

# Test user 148
print('USER 148 ----------')
collaborative_filtering_test_user_148 = collaborative_filtering_recommendation(148, movies_and_ratings)
print(collaborative_filtering_test_user_148)
already_seen_movies_148 = get_rated_movies_by_user(movies_and_ratings, 148)
print(already_seen_movies_148.loc[already_seen_movies_148['title'].isin(collaborative_filtering_test_user_148)])
similar_users_to_148 = display_similar_users_to_user_id(movies_and_ratings, 148)
print(similar_users_to_148)
print(check_ratings_by_similar_users(movies_and_ratings, 148, similar_users_to_148, collaborative_filtering_test_user_148))

# Test user 405
print('USER 405 ----------')
collaborative_filtering_test_user_405 = collaborative_filtering_recommendation(405, movies_and_ratings)
print(collaborative_filtering_test_user_405)
similar_users_to_405 = display_similar_users_to_user_id(movies_and_ratings, 405)
print(similar_users_to_405)
print(check_ratings_by_similar_users(movies_and_ratings, 405, similar_users_to_405, collaborative_filtering_test_user_405))

# Test user 418
print('USER 418 ----------')
collaborative_filtering_test_user_418 = collaborative_filtering_recommendation(418, movies_and_ratings)
print(collaborative_filtering_test_user_418)
similar_users_to_418 = display_similar_users_to_user_id(movies_and_ratings, 418)
print(similar_users_to_418)
print(check_ratings_by_similar_users(movies_and_ratings, 418, similar_users_to_418, collaborative_filtering_test_user_418))

USER 148 ----------
['Star Wars (1977)', 'Blade Runner (1982)', 'Raiders of the Lost Ark (1981)', 'Brazil (1985)', 'Fish Called Wanda, A (1988)']
       movie_id                           title release_date  \
6535         50                Star Wars (1977)  01-Jan-1977   
12268        89             Blade Runner (1982)  01-Jan-1982   
24841       174  Raiders of the Lost Ark (1981)  01-Jan-1981   
25066       175                   Brazil (1985)  01-Jan-1985   

       video_release_date                                           imdb_url  \
6535                  NaN  http://us.imdb.com/M/title-exact?Star%20Wars%2...   
12268                 NaN  http://us.imdb.com/M/title-exact?Blade%20Runne...   
24841                 NaN  http://us.imdb.com/M/title-exact?Raiders%20of%...   
25066                 NaN   http://us.imdb.com/M/title-exact?Brazil%20(1985)   

       genre_unknown  Action  Adventure  Animation  Children  ...  Musical  \
6535               0       1          1          0    

The tests above run for collaborative-filtering recommendation.  I tested the same three user id's as in the content-based recommendations: a median movie viewer (user 148 who has seen 65 movies), the user who has seen the most movies (user 405 with a grand total of 737 movies), and one of the users who has seen the least (user 418 who has seen only 20 movies).

User 148's top five movie recommendations are: 'Star Wars', 'Blade Runner', 'Raiders of the Lost Ark', 'Brazil', and 'A Fish Called Wanda'.

User 405's top five movie recommendations are: 'Pulp Fiction', "Schindler's List", 'The Shawshank Redemption', 'Raising Arizona', 'Monty Python and the Holy Grail'.

User 418's top five movie recommendations are: 'Titanic', 'Scream', 'Contact', 'Scream 2', 'Good Will Hunting'.